# Entropia degli intervalli sui tre corpora

Una misura di burstiness alternativa a $\sigma_\tau/\langle\tau\rangle$, costruita per
evitarne il difetto principale.

## Il problema che risolve

$\sigma_\tau/\langle\tau\rangle$ dipende dai primi due momenti di $p(\tau)$. Per code di
potenza con esponente $\mu<3$ la varianza **diverge**: lo stimatore non converge e cresce
col numero di eventi. È l'avvertenza (A) del notebook principale, e si vede nei dati —
per *prince* in *Guerra e pace* si passa da $1{,}30$ a $N=50\,000$ caratteri a $3{,}89$ sul
libro intero. Per questo tutti i confronti vanno fatti a lunghezza uguale, e i valori
assoluti non sono confrontabili con quelli del paper.

L'entropia della distribuzione degli intervalli non dipende da momenti divergenti.

## La definizione

Per ogni parola bersaglio si prendono gli intervalli $\tau_1,\dots,\tau_{n-1}$ fra
occorrenze consecutive e si definisce

$$J \;=\; \hat h\!\left(\ln \tau\right) \;-\; \hat h_{\rm Poisson}(n, \hat\mu)$$

dove $\hat h$ è l'entropia differenziale stimata e il secondo termine è la stessa quantità
calcolata su un processo di Poisson con lo **stesso tasso** e lo **stesso numero di
campioni**.

- $J > 0$ — $p(\tau)$ più dispersa di un'esponenziale: **bursty**
- $J = 0$ — processo di Poisson
- $J < 0$ — più regolare di Poisson

L'analogia con $\sigma_\tau/\langle\tau\rangle$ è diretta ($>1$, $=1$, $<1$), ma $J$ ha due
proprietà che quella non ha.

**Invarianza di scala esatta.** L'entropia differenziale è invariante per traslazione,
$h(Y+c)=h(Y)$, e una dilatazione dell'asse dei tempi $\tau \to c\,\tau$ diventa una
traslazione in scala logaritmica. Quindi $J$ è **immune** al confonditore della diversa
lunghezza delle frasi fra Wikipedia e Grokipedia (6,92 contro 4,38 frasi ogni 1000
caratteri), che altrimenti andrebbe discusso per ogni differenza osservata.

**Nessuna dipendenza da momenti divergenti.** La §3 lo verifica direttamente sulla stessa
scala di lunghezze su cui $\sigma_\tau/\langle\tau\rangle$ deriva.

## Lo stimatore e il controllo del bias

$\hat h$ è lo stimatore per spaziature di Vasicek: dati $y_{(1)}\le\dots\le y_{(n)}$,

$$\hat h_V \;=\; \frac{1}{n}\sum_{i=1}^{n} \ln\!\left[\frac{n}{2m}\left(y_{(i+m)}-y_{(i-m)}\right)\right],
\qquad m=\lfloor\sqrt{n}\rceil$$

con gli indici bloccati agli estremi. È **distorto verso il basso**, e la distorsione
dipende da $n$. Per questo il null non è la formula analitica ma una **simulazione con lo
stesso stimatore, la stessa $n$ e la stessa $\hat\mu$**: nella differenza $J$ il bias si
cancella. È la stessa logica dei null model A1/A2.

Gli intervalli sono interi (distanze in caratteri): prima della trasformazione logaritmica
si dequantizza con rumore uniforme in $[-\tfrac12,\tfrac12]$, e **la stessa dequantizzazione
si applica al null**, che viene quindi arrotondato agli interi per essere davvero parallelo.

## 1. Configurazione

In [ ]:
from pathlib import Path

QUI  = Path.cwd()
BASE = QUI if (QUI / "corpora_cache").is_dir() else QUI.parent
CACHE_WIKI = QUI / "cache_wikipedia"
CACHE_GROK = QUI / "cache_grokipedia"
CORPUS_V01 = QUI / "risultati_confronto" / "corpus_v01"
RIEPILOGO  = QUI / "risultati_confronto" / "dati" / "riepilogo.csv"
CACHE_LIB  = BASE / "corpora_cache"
OUT_DIR    = QUI / "risultati_entropia"

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"
START_PHRASE  = "Well, Prince, so Genoa and Lucca"
PROMPT_CHARS  = 8262

# stessi parametri del notebook a tre corpora, per poter accostare i numeri
N_EFF        = 60_000
ESCLUSI      = ["Buddhism"]     # v0.1 derivata da Wikipedia all'84%
N_SEG_LETT   = 20
MIN_EVENTS   = 15               # per la selezione dei bersagli
MIN_TAU      = 25               # sotto questa numerosita' J e' troppo rumoroso
N_NULL       = 40               # repliche del null di Poisson

N_LETTERS, N_FUNCTION, N_KEYWORDS, N_MATCHED = 19, 6, 7, 7
PROPER_CAP_RATIO = 0.6
RANDOM_SEED = 20260814
DPI = 150

In [ ]:
import re, math, json, ssl, gzip, zlib, hashlib, urllib.request
import html as htmlmod
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from scipy import stats as sps
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
try:
    import certifi
    HAS_CERTIFI = True
except Exception:
    HAS_CERTIFI = False

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": DPI, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True, "grid.alpha": .25,
                     "axes.axisbelow": True})
for d in (OUT_DIR, OUT_DIR / "figure", OUT_DIR / "dati"):
    d.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
print("scipy:", HAS_SCIPY, "| output ->", OUT_DIR.resolve())

## 2. Lo stimatore

`vasicek` implementa la formula sopra. `h_null` simula il processo di Poisson con lo stesso
$n$ e la stessa $\hat\mu$, arrotondato e dequantizzato come i dati veri, e restituisce la
media su `N_NULL` repliche. `J_intervalli` mette insieme le due cose.

In [ ]:
LOG2 = math.log(2.0)

def vasicek(y, m=None):
    """entropia differenziale per spaziature (Vasicek 1976), in nat"""
    y = np.sort(np.asarray(y, dtype=float))
    n = len(y)
    if n < 5:
        return np.nan
    if m is None:
        m = max(1, int(round(math.sqrt(n))))
    i = np.arange(n)
    hi = np.clip(i + m, 0, n - 1)
    lo = np.clip(i - m, 0, n - 1)
    d = y[hi] - y[lo]
    if np.any(d <= 0):
        return np.nan
    return float(np.mean(np.log(n * d / (2.0 * m))))

def _dequantizza(tau, r):
    """gli intervalli sono interi: si dequantizza prima del logaritmo"""
    t = np.asarray(tau, dtype=float) + r.uniform(-0.5, 0.5, size=len(tau))
    return np.log(np.clip(t, 1e-9, None))

def h_null(n, mu, r, reps=N_NULL):
    """entropia di ln(tau) per un processo di Poisson di media mu, stessa n.

    Il null viene arrotondato agli interi e dequantizzato esattamente come i dati:
    cosi' la distorsione dello stimatore, che dipende da n, si cancella in J."""
    out = []
    for _ in range(reps):
        t = np.maximum(1, np.round(r.exponential(mu, size=n))).astype(np.int64)
        v = vasicek(_dequantizza(t, r))
        if np.isfinite(v):
            out.append(v)
    return float(np.mean(out)) if out else np.nan

def J_intervalli(tau, r, min_tau=MIN_TAU, in_bit=True):
    """J = h(ln tau) - h_Poisson(n, mu).  >0 bursty, 0 Poisson, <0 regolare."""
    tau = np.asarray(tau, dtype=np.int64)
    tau = tau[tau > 0]
    n = len(tau)
    if n < min_tau:
        return np.nan
    h_obs = vasicek(_dequantizza(tau, r))
    if not np.isfinite(h_obs):
        return np.nan
    h_0 = h_null(n, float(tau.mean()), r)
    if not np.isfinite(h_0):
        return np.nan
    J = h_obs - h_0
    return J / LOG2 if in_bit else J

print("stimatore pronto (J espressa in bit)")

## 3. Validazione dello stimatore

Prima di applicarlo va verificato che faccia quello che dice, su processi di cui si conosce
la risposta. In particolare $J$ deve valere **zero** su un processo di Poisson: se non lo
facesse, la calibrazione del null sarebbe sbagliata.

In [ ]:
r = np.random.default_rng(RANDOM_SEED)
NCAMP = 60
print(f"J su processi sintetici (n = {{}} intervalli, {NCAMP} repliche)\n".format(200))

def prova(nome, gen, n=200, atteso=""):
    v = [J_intervalli(gen(n), r) for _ in range(NCAMP)]
    v = np.array([x for x in v if np.isfinite(x)])
    cv = []
    for _ in range(NCAMP):
        t = np.asarray(gen(n), float)
        cv.append(t.std(ddof=1) / t.mean())
    print(f"  {nome:34s} J = {v.mean():+.3f} +- {v.std():.3f}   "
          f"cv = {np.mean(cv):5.2f}   {atteso}")
    return v.mean()

MU = 1000.0
prova("Poisson (esponenziale)",
      lambda n: np.maximum(1, np.round(r.exponential(MU, n))).astype(int),
      atteso="<- deve dare J = 0")
prova("deterministico (tau costante)",
      lambda n: np.full(n, int(MU)),
      atteso="massima regolarita'")
prova("uniforme su [1, 2mu]",
      lambda n: r.integers(1, int(2*MU), n),
      atteso="piu' regolare di Poisson")
for s in (0.5, 1.0, 1.5, 2.0):
    prova(f"lognormale sigma={s}",
          lambda n, s=s: np.maximum(1, np.round(
              r.lognormal(math.log(MU) - s*s/2, s, n))).astype(int),
          atteso="dispersione crescente")
for mu_pl in (3.5, 2.4, 2.0):
    def pareto(n, a=mu_pl-1):
        return np.maximum(1, np.round(MU * (r.pareto(a, n) + 1) * (a-1)/a
                                      if a > 1 else MU*(r.pareto(a, n)+1))).astype(int)
    prova(f"legge di potenza mu={mu_pl}", pareto,
          atteso="coda larga" + ("  (mu del paper)" if mu_pl == 2.4 else ""))

In [ ]:
# --- J e' davvero invariante per dilatazione dell'asse dei tempi? ---
r = np.random.default_rng(RANDOM_SEED)
base = np.maximum(1, np.round(r.lognormal(math.log(1000) - 0.5, 1.0, 300))).astype(int)
print("stessa sequenza di intervalli, riscalata di un fattore c:\n")
print(f"  {'c':>6s} {'<tau>':>9s} {'cv_tau':>8s} {'J (bit)':>9s}")
for c in (0.25, 0.5, 1, 2, 4):
    t = np.maximum(1, np.round(base * c)).astype(int)
    cv = t.std(ddof=1) / t.mean()
    print(f"  {c:>6g} {t.mean():>9.1f} {cv:>8.3f} {J_intervalli(t, r):>9.3f}")
print("\n  cv_tau e J sono entrambe invarianti per riscalatura: atteso.")
print("  La differenza fra le due emerge sulla LUNGHEZZA del campione, non sulla scala.")

## 4. La verifica che conta: stabilità con la lunghezza del testo

Qui si mette alla prova la ragione per cui questa misura è stata introdotta. Si prende
*Guerra e pace*, si misurano $\sigma_\tau/\langle\tau\rangle$ e $J$ per *prince* su
sottocampioni annidati di lunghezza crescente, e si guarda quale delle due deriva.

In [ ]:
GUT_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
GUT_END   = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
STRUCT = re.compile(r"^[ \t]*(?:BOOK\s+[A-Z]+[^\n]*|CHAPTER\s+[IVXLCDM\d]+[^\n]*|"
                    r"(?:FIRST|SECOND)\s+EPILOGUE[^\n]*|EPILOGUE[^\n]*|CONTENTS[^\n]*|"
                    r"PART\s+[IVXLCDM\d]+[^\n]*|APPENDIX[^\n]*|\d+)[ \t]*$", re.M)
WORD_CHAR = r"[^\W\d_]"
TOKEN_RE  = re.compile(WORD_CHAR + r"+(?:['\u2019\-]" + WORD_CHAR + r"+)*", re.UNICODE)

def contesto_ssl():
    if HAS_CERTIFI:
        try: return ssl.create_default_context(cafile=certifi.where())
        except Exception: pass
    try: return ssl.create_default_context()
    except ssl.SSLError:
        c = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        c.check_hostname = False; c.verify_mode = ssl.CERT_NONE
        return c
CTX = contesto_ssl()

def libro():
    fn = CACHE_LIB / (hashlib.sha256(GUTENBERG_URL.encode()).hexdigest()[:12] + ".txt")
    if fn.exists():
        raw = fn.read_bytes().decode("utf-8", errors="replace")
        if "\r\r" not in raw:
            return raw
    req = urllib.request.Request(GUTENBERG_URL, headers={"User-Agent": "ricerca/1.0"})
    with urllib.request.urlopen(req, timeout=90, context=CTX) as rr:
        raw = rr.read().decode("utf-8", errors="replace")
    CACHE_LIB.mkdir(parents=True, exist_ok=True)
    fn.write_bytes(raw.encode("utf-8"))
    return raw

raw = libro().replace("\r\n", "\n").replace("\r", "\n")
raw = raw[GUT_START.search(raw).end():]
raw = raw[:GUT_END.search(raw).start()]
raw = raw[raw.find(START_PHRASE):]
raw = STRUCT.sub("", raw)
raw = re.sub(r"\n[ \t]+\n", "\n\n", raw)
BODY = re.sub(r"\n{3,}", "\n\n", raw).strip()[PROMPT_CHARS:]
print(f"corpo letterario: {len(BODY):,} caratteri")

_cache_re = {}
def pos_parola(text, w):
    if w not in _cache_re:
        _cache_re[w] = re.compile(r"(?<!" + WORD_CHAR + r")" + re.escape(w) +
                                  r"(?!" + WORD_CHAR + r")", re.IGNORECASE | re.UNICODE)
    return np.fromiter((m.start() for m in _cache_re[w].finditer(text)), dtype=np.int64)

r = np.random.default_rng(RANDOM_SEED)
righe = []
for N in [20_000, 40_000, 60_000, 100_000, 200_000, 400_000, 800_000, len(BODY)]:
    N = min(N, len(BODY))
    seg = BODY[:N]
    for w in ["prince", "pierre", "e"]:
        pos = (pos_parola(seg, w) if len(w) > 1
               else np.flatnonzero(np.array(list(seg.lower())) == w))
        if len(pos) < MIN_TAU + 1:
            continue
        tau = np.diff(pos)
        righe.append(dict(N=N, parola=w, n_eventi=len(pos),
                          cv_tau=tau.std(ddof=1)/tau.mean(),
                          J=J_intervalli(tau, r)))
FS = pd.DataFrame(righe)
FS.to_csv(OUT_DIR / "dati" / "stabilita_lunghezza.csv", index=False)
for w in FS["parola"].unique():
    g = FS[FS["parola"] == w].sort_values("N")
    r0, r1 = g.iloc[0], g.iloc[-1]
    print(f"\n  '{w}'  da N={r0['N']:,} a N={r1['N']:,}")
    print(f"    cv_tau : {r0['cv_tau']:.2f} -> {r1['cv_tau']:.2f}   "
          f"(fattore {r1['cv_tau']/r0['cv_tau']:.2f})")
    print(f"    J      : {r0['J']:+.3f} -> {r1['J']:+.3f}   "
          f"(variazione {r1['J']-r0['J']:+.3f} bit)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
for w, c in zip(FS["parola"].unique(), ["#0072B2", "#D55E00", "#009E73"]):
    g = FS[FS["parola"] == w].sort_values("N")
    et = f'lettera "{w}"' if len(w) == 1 else f'"{w}"'
    axes[0].plot(g["N"], g["cv_tau"], "o-", color=c, ms=5, label=et)
    axes[1].plot(g["N"], g["J"], "o-", color=c, ms=5, label=et)
for ax, ylab, tit in [(axes[0], r"$\sigma_\tau/\langle\tau\rangle$",
                       "A) la burstiness classica deriva con $N$"),
                      (axes[1], "$J$ [bit]", "B) $J$ e' stabile")]:
    ax.set_xscale("log"); ax.set_xlabel("$N$ [caratteri]"); ax.set_ylabel(ylab)
    ax.axvline(N_EFF, color="crimson", ls=":", lw=1.6)
    ax.set_title(tit, fontsize=10); ax.legend(fontsize=8)
axes[0].axhline(1, color="grey", ls=":", lw=1)
axes[1].axhline(0, color="grey", ls=":", lw=1)
fig.suptitle("Stabilita' dei due indicatori con la lunghezza del testo\n"
             "(Guerra e pace; linea rossa = N_EFF usato nei confronti)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "stabilita.png"); plt.show()

## 5. I tre corpora

In [ ]:
APPARATO = re.compile(r"^==+\s*(See also|References|Notes|Citations|Sources|Bibliography|"
                      r"Further reading|External links|Works cited|Footnotes|"
                      r"Explanatory notes|General sources)\s*==+\s*$", re.M | re.I)
INTEST = re.compile(r"^==+.*?==+\s*$", re.M)
TTS = re.compile(r'<span[^>]*data-tts-block="true"[^>]*>(.*?)</span>', re.S)
ARTICOLO = re.compile(r"<article[^>]*>(.*?)</article>", re.S)

def _frag(h):
    h = re.sub(r"(?is)<(script|style|button|svg|nav|footer|aside)[^>]*>.*?</\1>", " ", h)
    h = re.sub(r"<[^>]+>", " ", h)
    h = htmlmod.unescape(h)
    h = re.sub(r"\[\d+\]", " ", h)
    h = re.sub(r"[ \t\u00a0]+", " ", h)
    return re.sub(r"\n\s*\n+", "\n\n", h).strip()

def safe(t):
    return re.sub(r"[^A-Za-z0-9_]", "", t.replace(" ", "_"))[:60]

def carica_wiki(t):
    fn = CACHE_WIKI / (hashlib.sha256(t.encode()).hexdigest()[:14] + ".txt")
    if not fn.exists(): return ""
    tx = fn.read_bytes().decode("utf-8", errors="replace")
    m = APPARATO.search(tx)
    if m: tx = tx[:m.start()]
    tx = INTEST.sub("", tx)
    tx = re.sub(r"\n[ \t]+\n", "\n\n", tx)
    return re.sub(r"\n{3,}", "\n\n", tx).strip()

def carica_grok(t):
    fn = CACHE_GROK / (safe(t) + ".html")
    if not fn.exists(): return ""
    p = fn.read_bytes().decode("utf-8", errors="replace")
    b = [_frag(x) for x in TTS.findall(p)]
    b = [x for x in b if len(x) > 40]
    if b: return "\n\n".join(b)
    m = ARTICOLO.search(p)
    return _frag(m.group(1)) if m else ""

def carica_v01(t):
    fn = CORPUS_V01 / (safe(t) + ".txt")
    return fn.read_bytes().decode("utf-8", errors="replace") if fn.exists() else ""

R = pd.read_csv(RIEPILOGO)
TITOLI = [t for t in R["titolo"] if t not in ESCLUSI]
CORPORA = {}
for nome, f in [("wikipedia", carica_wiki), ("grok_v01", carica_v01),
                ("grok_oggi", carica_grok)]:
    CORPORA[nome] = {t: f(t) for t in TITOLI}
ok = [t for t in TITOLI if all(len(CORPORA[c].get(t, "")) >= N_EFF for c in CORPORA)]
print(f"titoli: {len(TITOLI)} (esclusi {ESCLUSI}) | terne complete a N_EFF={N_EFF:,}: {len(ok)}")
TITOLI = ok

starts = np.linspace(0, max(len(BODY) - N_EFF, 0), N_SEG_LETT).astype(int)
CORPORA["letterario"] = {f"wrnpc_{i}": BODY[s:s+N_EFF] for i, s in enumerate(starts)}
print(f"riferimento letterario: {len(CORPORA['letterario'])} segmenti da {N_EFF:,}")

In [ ]:
STOPWORDS = set("""a about above after again against all am an and any are as at be because been
before being below between both but by can cannot could did do does doing down during each few for
from further had has have having he her here hers herself him himself his how i if in into is it its
itself me more most my myself no nor not of off on once only or other ought our ours ourselves out
over own same she should so some such than that the their theirs them themselves then there these
they this those through to too under until up very was we were what when where which while who whom
why with would you your yours yourself yourselves said one two would shall may might must upon""".split())
VOWELS = set("aeiou"); SENT_END = set(".!?")

def word_stats(text):
    freq, cap, tot = Counter(), Counter(), Counter()
    pe, primo = 0, True
    for m in TOKEN_RE.finditer(text):
        w = m.group(0); lw = w.lower(); freq[lw] += 1
        gap = text[pe:m.start()]
        if not (primo or any(c in SENT_END for c in gap) or "\n\n" in gap):
            tot[lw] += 1
            if w[0].isupper(): cap[lw] += 1
        pe, primo = m.end(), False
    return freq, {w: (cap[w]/tot[w] if tot[w] >= 3 else 0.0) for w in freq}

def select_targets(text):
    low = text.lower()
    letters = [c for c, _ in Counter(c for c in low if c.isalpha() and c.isascii())
               .most_common(N_LETTERS)]
    freq, capr = word_stats(text)
    ordered = [w for w, _ in freq.most_common() if len(w) >= 2]
    funcs = [w for w in ordered if w in STOPWORDS][:N_FUNCTION]
    isp = lambda w: capr.get(w, 0) >= PROPER_CAP_RATIO
    isk = lambda w: isp(w) or (w not in STOPWORDS and len(w) >= 4)
    keys = [w for w in ordered if isk(w)][:N_KEYWORDS]
    ks = set(keys)
    pool = [w for w in ordered if w not in ks and not isp(w) and w not in funcs]
    matched, used = [], set()
    for k in keys[:N_MATCHED]:
        fk = freq[k]
        cand = sorted((w for w in pool if w not in used),
                      key=lambda w: (abs(math.log((freq[w]+1e-9)/(fk+1e-9))), w))
        if cand: matched.append(cand[0]); used.add(cand[0])
    return ([("vocali", "vocali", VOWELS)] +
            [("spazio", "spazio", " ")] +
            [(c, "lettera", c) for c in letters] +
            [(w, "funzione", w) for w in funcs] +
            [(w, "keyword", w) for w in keys] +
            [(w, "appaiata", w) for w in matched])

def posizioni(text, carr, tipo, chiave):
    if tipo == "vocali":  return np.flatnonzero(np.isin(carr, list(chiave)))
    if tipo == "spazio":  return np.flatnonzero(carr == " ")
    if tipo == "lettera": return np.flatnonzero(carr == chiave)
    return pos_parola(text, chiave)

REC = []
for corpus, docs in CORPORA.items():
    print(f"{corpus:12s} ", end="", flush=True)
    for lab, testo in docs.items():
        t = testo[:N_EFF]
        carr = np.array(list(t.lower()))
        rr = np.random.default_rng(int(hashlib.sha256(
            f"{corpus}|{lab}".encode()).hexdigest()[:8], 16) ^ RANDOM_SEED)
        for etichetta, tipo, chiave in select_targets(t):
            pos = posizioni(t, carr, tipo, chiave)
            if len(pos) < MIN_EVENTS:
                continue
            tau = np.diff(pos)
            REC.append(dict(corpus=corpus, titolo=lab, sequenza=etichetta, tipo=tipo,
                            n_eventi=len(pos), mean_tau=float(tau.mean()),
                            cv_tau=float(tau.std(ddof=1)/tau.mean()),
                            J=J_intervalli(tau, rr)))
        print(".", end="", flush=True)
    print(" fatto")

A = pd.DataFrame(REC)
A.to_csv(OUT_DIR / "dati" / "entropia_sequenze.csv", index=False)
print(f"\nrecord: {len(A):,} | con J stimabile: {A['J'].notna().sum():,} "
      f"({A['J'].notna().mean():.0%}, soglia MIN_TAU={MIN_TAU})")
print(A.groupby("tipo")["J"].apply(lambda s: f"{s.notna().mean():.0%}").to_string())

## 6. Risultati per corpus

In [ ]:
ORD = ["letterario", "wikipedia", "grok_v01", "grok_oggi"]
NOMI = {"letterario": "Guerra e pace", "wikipedia": "Wikipedia",
        "grok_v01": "Grokipedia v0.1", "grok_oggi": "Grokipedia oggi"}
LIV = [("lettera", "lettere"), ("funzione", "parole funzione"),
       ("appaiata", "controlli appaiati"), ("keyword", "keyword")]

def per_doc(corpus, tipo, col):
    s = A[(A["corpus"] == corpus) & (A["tipo"] == tipo)]
    return s.groupby("titolo")[col].mean().dropna()

print(f"J [bit] per livello — media fra testi, N = {N_EFF:,} caratteri")
print("(J = 0 e' il processo di Poisson; J > 0 = intervalli piu' dispersi)\n")
print(f"  {'livello':20s}" + "".join(f"{NOMI[c]:>18s}" for c in ORD))
righe = []
for tipo, nome in LIV:
    riga = f"  {nome:20s}"
    for c in ORD:
        v = per_doc(c, tipo, "J")
        riga += f"{v.mean():>11.3f}+-{v.std():<5.3f}" if len(v) else f"{'n.d.':>18s}"
        righe.append(dict(tipo=tipo, corpus=c, J=v.mean(), J_sd=v.std(), n=len(v),
                          cv=per_doc(c, tipo, "cv_tau").mean()))
    print(riga)
LIVELLI = pd.DataFrame(righe)
LIVELLI.to_csv(OUT_DIR / "dati" / "livelli.csv", index=False)

print(f"\nPer confronto, sigma_tau/<tau> sugli stessi dati")
print(f"  {'livello':20s}" + "".join(f"{NOMI[c]:>18s}" for c in ORD))
for tipo, nome in LIV:
    riga = f"  {nome:20s}"
    for c in ORD:
        v = per_doc(c, tipo, "cv_tau")
        riga += f"{v.mean():>18.3f}" if len(v) else f"{'n.d.':>18s}"
    print(riga)

## 7. Confronti appaiati per titolo

In [ ]:
COPPIE = [("grok_oggi", "wikipedia"), ("grok_v01", "wikipedia"),
          ("grok_oggi", "grok_v01")]
righe = []
for a, b in COPPIE:
    for tipo, nome in LIV:
        x, y = per_doc(a, tipo, "J"), per_doc(b, tipo, "J")
        i = x.index.intersection(y.index)
        if len(i) < 6: continue
        d = (x[i] - y[i]).values
        p = float(sps.wilcoxon(x[i], y[i]).pvalue) if HAS_SCIPY else np.nan
        righe.append(dict(confronto=f"{NOMI[a]} vs {NOMI[b]}", livello=nome, n=len(i),
                          diff_mediana=float(np.median(d)), vince_primo=float((d > 0).mean()),
                          p=p))
PT = pd.DataFrame(righe)
PT.to_csv(OUT_DIR / "dati" / "confronti_appaiati.csv", index=False)
print("Differenze di J, appaiate per titolo (Wilcoxon)\n")
print(f"  {'confronto':36s}{'livello':20s}{'n':>4s}{'diff':>9s}{'vince 1o':>10s}{'p':>10s}")
for _, x in PT.iterrows():
    st = "" if not np.isfinite(x["p"]) else (" ***" if x["p"] < 1e-3 else
         " **" if x["p"] < 1e-2 else " *" if x["p"] < .05 else "")
    print(f"  {x['confronto']:36s}{x['livello']:20s}{int(x['n']):>4d}"
          f"{x['diff_mediana']:>+9.3f}{x['vince_primo']:>9.0%}{x['p']:>10.1e}{st}")

## 8. Cosa aggiunge $J$ rispetto a $\sigma_\tau/\langle\tau\rangle$

Se le due misurassero la stessa cosa, sarebbero ridondanti. La correlazione dice quanta
informazione condividono; i casi in cui divergono dicono dove $J$ è più informativa.

In [ ]:
K = A[(A["tipo"] == "keyword") & A["J"].notna() & A["cv_tau"].notna()]
if HAS_SCIPY and len(K) > 20:
    rho, p = sps.spearmanr(K["cv_tau"], K["J"])
    print(f"Spearman(cv_tau, J) su {len(K):,} sequenze keyword: rho = {rho:+.3f}  p = {p:.1e}")
    print(f"  -> condividono circa il {100*rho**2:.0f}% della varianza di rango")

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.7))
COL = {"letterario": "#333333", "wikipedia": "#0072B2",
       "grok_v01": "#E69F00", "grok_oggi": "#D55E00"}
ax = axes[0]
for c in ORD:
    s = K[K["corpus"] == c]
    ax.scatter(s["cv_tau"], s["J"], s=14, alpha=.55, color=COL[c], label=NOMI[c])
ax.axhline(0, color="grey", ls=":", lw=1); ax.axvline(1, color="grey", ls=":", lw=1)
ax.set_xlabel(r"$\sigma_\tau/\langle\tau\rangle$"); ax.set_ylabel("$J$ [bit]")
ax.set_title("A) le due misure a confronto (keyword)", fontsize=10)
ax.legend(fontsize=7)

ax = axes[1]
xs = np.arange(len(LIV))
for i, c in enumerate(ORD):
    v = [LIVELLI[(LIVELLI.tipo == t) & (LIVELLI.corpus == c)]["J"].iloc[0] for t, _ in LIV]
    e = [LIVELLI[(LIVELLI.tipo == t) & (LIVELLI.corpus == c)]["J_sd"].iloc[0] for t, _ in LIV]
    ax.errorbar(xs + .05*(i-1.5), v, yerr=e, marker="os^D"[i], ms=7, lw=1.8,
                capsize=3, color=COL[c], label=NOMI[c])
ax.axhline(0, color="grey", ls=":", lw=1.2)
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
ax.set_ylabel("$J$ [bit]"); ax.set_title("B) $J$ per livello linguistico", fontsize=10)
ax.legend(fontsize=7)

ax = axes[2]
dati, et = [], []
for c in ORD:
    v = per_doc(c, "keyword", "J")
    if len(v): dati.append(v.values); et.append(NOMI[c].replace(" ", "\n"))
if dati:
    bp = ax.boxplot(dati, tick_labels=et, showmeans=True, widths=.55, patch_artist=True)
    for b, c in zip(bp["boxes"], ORD): b.set(facecolor=COL[c], alpha=.3)
    for i, v in enumerate(dati):
        ax.scatter(np.full(len(v), i+1) + rng.normal(0, .05, len(v)), v,
                   s=14, color="k", alpha=.5, zorder=4)
ax.axhline(0, color="grey", ls=":", lw=1.2)
ax.set_ylabel("$J$ keyword [bit]"); ax.tick_params(axis="x", labelsize=7.5)
ax.set_title("C) distribuzione per documento", fontsize=10)

fig.suptitle(f"Entropia degli intervalli sui tre corpora  "
             f"(N = {N_EFF:,} caratteri, {len(TITOLI)} terne)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "entropia_intervalli.png"); plt.show()

## 9. Sintesi

In [ ]:
L = []
L.append("ENTROPIA DEGLI INTERVALLI — J = h(ln tau) - h_Poisson(n, mu)")
L.append("=" * 68)
L.append(f"terne: {len(TITOLI)} | N_EFF = {N_EFF:,} caratteri | MIN_TAU = {MIN_TAU}")
L.append(f"sequenze con J stimabile: {A['J'].notna().sum():,}/{len(A):,} "
         f"({A['J'].notna().mean():.0%})")
L.append("")
L.append("VALIDAZIONE")
for w in FS["parola"].unique():
    g = FS[FS["parola"] == w].sort_values("N")
    a, b = g.iloc[0], g.iloc[-1]
    L.append(f"  '{w}': da N={a['N']:,} a N={b['N']:,}  "
             f"cv_tau {a['cv_tau']:.2f}->{b['cv_tau']:.2f} (x{b['cv_tau']/a['cv_tau']:.2f})  "
             f"J {a['J']:+.2f}->{b['J']:+.2f} ({b['J']-a['J']:+.2f} bit)")
L.append("")
L.append("J PER LIVELLO [bit]")
for tipo, nome in LIV:
    s = LIVELLI[LIVELLI["tipo"] == tipo].set_index("corpus")
    L.append(f"  {nome:20s}" + "  ".join(
        f"{NOMI[c]}={s.loc[c,'J']:+.3f}" for c in ORD if c in s.index))
L.append("")
L.append("CONFRONTI APPAIATI (keyword)")
for _, x in PT[PT["livello"] == "keyword"].iterrows():
    sig = "significativa" if x["p"] < .05 else "non significativa"
    L.append(f"  {x['confronto']:36s} diff {x['diff_mediana']:+.3f} bit, "
             f"p={x['p']:.1e} ({sig})")
L.append("")
L.append("NOTE")
L.append("  - J e' invariante per dilatazione dell'asse dei caratteri: il confonditore")
L.append("    della diversa lunghezza delle frasi non la tocca.")
L.append("  - il null e' simulato con lo stesso stimatore, n e mu: il bias si cancella.")
L.append(f"  - sotto MIN_TAU={MIN_TAU} intervalli J non viene stimata.")
sintesi = "\n".join(L)
(OUT_DIR / "sintesi.txt").write_text(sintesi, encoding="utf-8")
print(sintesi)